In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier, DMatrix

In [2]:
RANDOM_STATE = 12345

# Load data

In [3]:
breast_cancer = load_breast_cancer(as_frame=True)
breast_cancer_df = breast_cancer.frame
feature_cols = [i.replace(' ', '_') for i in breast_cancer.feature_names]
target_cols = ['target']
breast_cancer_df.columns = feature_cols + target_cols
breast_cancer_df.head()

,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
X = breast_cancer_df[feature_cols].values
y = breast_cancer_df[target_cols].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=RANDOM_STATE)

print('Train', X_train.shape, y_train.shape)
print('Valid', X_valid.shape, y_valid.shape)
print('Test', X_test.shape, y_test.shape)

Train (455, 30) (455,)
Valid (57, 30) (57,)
Test (57, 30) (57,)


# XGBClassifier

In [5]:
params = {
    'booster': 'gbtree',
    'objective': 'binary:logistic',   # logistic regression for binary classification, output probability
    'learning_rate': 0.3,
    'max_depth': 3,
    'n_estimators': 3,
    'reg_lambda': 1,                  # default for gbtree booster
    'gamma': 0,
    'min_child_weight': 1,
    'tree_method': 'exact',           # Exact greedy algorithm. Enumerates all split candidates.
    'random_state': RANDOM_STATE,
}


# Instantiate XGBClassifier with parameters
classifier = XGBClassifier(**params)

# Train the model using fit() method
classifier.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,'gbtree'
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [6]:
booster = classifier.get_booster()
tree_dump = booster.get_dump(with_stats=True)

In [7]:
print(tree_dump[0])

0:[f22<105.150002] yes=1,no=2,missing=1,gain=324.59375,cover=106.483513
	1:[f27<0.135199994] yes=3,no=4,missing=3,gain=15.0008545,cover=63.8901062
		3:[f10<0.542100012] yes=7,no=8,missing=7,gain=3.24246216,cover=60.1456299
			7:leaf=0.47096166,cover=58.9754829
			8:leaf=-0.0182293858,cover=1.17014849
		4:[f16<0.0480500013] yes=9,no=10,missing=9,gain=4.98022175,cover=3.74447513
			9:leaf=-0.32650432,cover=2.57432675
			10:leaf=0.258249402,cover=1.17014849
	2:[f7<0.0488649979] yes=5,no=6,missing=5,gain=31.7784576,cover=42.5934029
		5:[f17<0.00988649949] yes=11,no=12,missing=11,gain=9.50725746,cover=5.85074234
			11:leaf=-0.307456344,cover=3.04238605
			12:leaf=0.353185147,cover=2.80835629
		6:[f26<0.223700002] yes=13,no=14,missing=13,gain=6.57095337,cover=36.7426605
			13:leaf=-0.0182293858,cover=1.17014849
			14:leaf=-0.756377816,cover=35.5725136



In [8]:
print(tree_dump[1])

0:[f20<16.7950001] yes=1,no=2,missing=1,gain=172.427231,cover=99.1552429
	1:[f27<0.142349988] yes=3,no=4,missing=3,gain=28.0716629,cover=61.5892639
		3:[f13<46.625] yes=7,no=8,missing=7,gain=1.99213409,cover=55.0839272
			7:leaf=0.394367576,cover=53.9241142
			8:leaf=-0.00158019224,cover=1.15980971
		4:[f4<0.0913649946] yes=9,no=10,missing=9,gain=6.33043432,cover=6.50533915
			9:leaf=0.25702402,cover=1.10231447
			10:leaf=-0.369930714,cover=5.40302467
	2:[f11<0.492299974] yes=5,no=6,missing=5,gain=7.67210388,cover=37.565979
		5:leaf=0.155011296,cover=1.18419266
		6:[f26<0.190699995] yes=11,no=12,missing=11,gain=3.28011322,cover=36.3817863
			11:leaf=-0.00914661773,cover=1.15803051
			12:leaf=-0.530088425,cover=35.2237549



In [9]:
print(tree_dump[2])

0:[f22<102.650002] yes=1,no=2,missing=1,gain=106.497108,cover=84.9108658
	1:[f24<0.17825] yes=3,no=4,missing=3,gain=4.1000824,cover=42.944313
		3:[f12<4.11849976] yes=7,no=8,missing=7,gain=2.70004272,cover=41.8795204
			7:leaf=0.366287231,cover=40.8531494
			8:leaf=-0.0686659813,cover=1.02636909
		4:leaf=-0.140089035,cover=1.06479549
	2:[f7<0.0488649979] yes=5,no=6,missing=5,gain=13.1170692,cover=41.9665489
		5:[f21<27.6500015] yes=9,no=10,missing=9,gain=5.68063211,cover=6.34849024
			9:leaf=0.270545006,cover=3.78226662
			10:leaf=-0.231468886,cover=2.56622362
		6:[f21<20.6450005] yes=11,no=12,missing=11,gain=5.66519547,cover=35.6180573
			11:leaf=-0.00906199962,cover=3.18481636
			12:leaf=-0.430692434,cover=32.4332428



# Algorithm
**Initialize with a Baseline Prediction ($F_0$)**  
Start with a constant prediction that minimizes the overall loss function.  
For **log loss**, we will update the logit and then convert it to a probability using the sigmoid function.  
So this constant is the **mean of the log target values**:
$$p_0 = \frac{1}{n}\sum_{i=1}^{n} y_i$$
$$F_0 = log(\frac{p_0}{1-p_0})$$

In [10]:
logit0 = np.log(y_train.mean() / (1 - y_train.mean()))
F = F0 = np.full(len(y_train), logit0)
p0 = 1 / (1 + np.exp(-F0))

**Calculate Pseudo-Residuals:**  
Compute the negative gradient of the loss function with respect to the current predictions.
$$ PseudoResiduals_i = - (predict_i - observe_i) = observe_i - predict_i $$

In [11]:
residuals = y_train - p0

**Fit a New Tree:**  
Train a decision tree to predict these pseudo-residuals.

In [12]:
def similarity_score(residual, prev_prob, reg_lambda=1):
    return np.sum(residual)**2/(np.sum(prev_prob*(1-prev_prob)) + reg_lambda)

def leaf_value(residual, prev_prob, reg_lambda=1, learning_rate=0.3):
    return learning_rate*np.sum(residual)/(np.sum(prev_prob*(1-prev_prob)) + reg_lambda)
    
def find_best_split(X, residuals, feature_cols, prev_prob, min_child_weight=1):
    
    similarity_score_parent = similarity_score(residuals, prev_prob)

    max_gain = -1
    best_feature_idx = None
    best_feature = None
    best_thres = None
    best_left_grp=None
    best_right_grp=None
    for idx in range(len(feature_cols)):
        # For each feature, sort unique values
        Xf = X[:, idx]
        sort_unq_Xf = np.sort(np.unique(Xf))

        # find possible threshold between consecutive sorted values
        possible_thresholds = (sort_unq_Xf[:-1] + sort_unq_Xf[1:])/2 

        for thres in possible_thresholds:
            # For each possible threshold, splitting the node into a "left" group and a "right" group
            mask_left = Xf <= thres
            mask_right = ~mask_left

            # compute cover and if cover < min_child_weight, then skip.
            HL = np.sum(prev_prob[mask_left] * (1 - prev_prob[mask_left]))
            HR = np.sum(prev_prob[mask_right] * (1 - prev_prob[mask_right]))
            if HL < min_child_weight or HR < min_child_weight:
                continue   # XGBoost would reject this split too

            similarity_score_left_grp = similarity_score(residuals[mask_left], prev_prob[mask_left])
            similarity_score_right_grp = similarity_score(residuals[mask_right], prev_prob[mask_right])
    
            gain = similarity_score_left_grp + similarity_score_right_grp - similarity_score_parent
    
            if (gain > max_gain) and (gain > 0):
                max_gain = gain
                best_feature_idx = idx
                best_feature = feature_cols[idx]
                best_thres = thres
                best_left_grp = [X[mask_left,:], residuals[mask_left], prev_prob[mask_left]]
                best_right_grp = [X[mask_right,:], residuals[mask_right], prev_prob[mask_right]]
                

    return best_feature, best_feature_idx, best_thres, best_left_grp, best_right_grp, max_gain

In [13]:
f, fidx, thres, left_grp, right_grp, gain = find_best_split(X_train, residuals, feature_cols, p0)
print(f'0:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(left_grp[0], left_grp[1], feature_cols, left_grp[2])
print(f'	1:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols, left_grp10[2])
print(f'		3:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			7:leaf={leaf_value(left_grp100[1], left_grp100[2])}')
print(f'			8:leaf={leaf_value(right_grp100[1], right_grp100[2])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols, right_grp10[2])
print(f'		4:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			9:leaf={leaf_value(left_grp101[1], left_grp101[2])}')
print(f'			10:leaf={leaf_value(right_grp101[1], right_grp101[2])}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(right_grp[0], right_grp[1], feature_cols, right_grp[2])
print(f'	2:f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols, left_grp10[2])
print(f'		5:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			11:leaf={leaf_value(left_grp100[1], left_grp100[2])}')
print(f'			12:leaf={leaf_value(right_grp100[1], right_grp100[2])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols, right_grp10[2])
print(f'		6:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			13:leaf={leaf_value(left_grp101[1], left_grp101[2])}')
print(f'			14:leaf={leaf_value(right_grp101[1], right_grp101[2])}')

0:[f22<=105.15000000], gain=324.5937036435435
	1:[f27<=0.13520000], gain=15.000851728713144
		3:[f10<=0.54210000], gain=3.2424787554473937
			7:leaf=0.47096164958915315
			8:leaf=-0.01822936953981414
		4:[f16<=0.04805000], gain=4.980221120303
			9:leaf=-0.32650427379303354
			10:leaf=0.2582494018140337
	2:f7<=0.04886500], gain=31.778468259408555
		5:[f17<=0.00988650], gain=9.507257087333429
			11:leaf=-0.3074563106796116
			12:leaf=0.3531851476044012
		6:[f26<=0.22370000], gain=6.5709462837282615
			13:leaf=-0.01822936953981414
			14:leaf=-0.7563777624423408


**Update Predictions:**  
Add the new tree's scaled leaf outputs to the current predictions:

 $$F_m(x) = F_{m-1}(x) + \eta f_m(x)$$

 where $\eta$ is the **learning rate** and $f_m(x)$ is the newly trained tree.

In [14]:
F1 = booster.predict(DMatrix(X_train), iteration_range=(0, 1)) #[start, end)

**Repeat Until the Stopping Condition Is Met at n_estimators=3**  

In [15]:
p1 = F1
residuals = y_train - F1

f, fidx, thres, left_grp, right_grp, gain = find_best_split(X_train, residuals, feature_cols, p1)
print(f'0:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(left_grp[0], left_grp[1], feature_cols, left_grp[2])
print(f'	1:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols, left_grp10[2])
print(f'		3:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			7:leaf={leaf_value(left_grp100[1], left_grp100[2])}')
print(f'			8:leaf={leaf_value(right_grp100[1], right_grp100[2])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols, right_grp10[2])
print(f'		4:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			9:leaf={leaf_value(left_grp101[1], left_grp101[2])}')
print(f'			10:leaf={leaf_value(right_grp101[1], right_grp101[2])}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(right_grp[0], right_grp[1], feature_cols, right_grp[2])
print(f'	2:f{fidx}<={thres:.8f}], gain={gain}')
print(f'		5:leaf={leaf_value(left_grp10[1], left_grp10[2])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols, right_grp10[2])
print(f'		6:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			13:leaf={leaf_value(left_grp101[1], left_grp101[2])}')
print(f'			14:leaf={leaf_value(right_grp101[1], right_grp101[2])}')

0:[f20<=16.79500000], gain=172.427250371855
	1:[f27<=0.14235000], gain=28.0716605778438
		3:[f13<=46.62500000], gain=1.9921330287386922
			7:leaf=0.3943675989854442
			8:leaf=-0.0015801920299586074
		4:[f4<=0.09136500], gain=6.330434697991046
			9:leaf=0.25702400874327747
			10:leaf=-0.3699307456728472
	2:f11<=0.49230000], gain=7.672095655612182
		5:leaf=0.15501129331980376
		6:[f26<=0.19070000], gain=3.280123086547647
			13:leaf=-0.00914661773004858
			14:leaf=-0.5300884059822512


In [16]:
F2 = booster.predict(DMatrix(X_train), iteration_range=(0, 2)) #[start, end)

In [19]:
p2 = F2
residuals = y_train - F2

f, fidx, thres, left_grp, right_grp, gain = find_best_split(X_train, residuals, feature_cols, p2)
print(f'0:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(left_grp[0], left_grp[1], feature_cols, left_grp[2])
print(f'	1:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols, left_grp10[2])
print(f'		3:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			7:leaf={leaf_value(left_grp100[1], left_grp100[2])}')
print(f'			8:leaf={leaf_value(right_grp100[1], right_grp100[2])}')

print(f'		4:leaf={leaf_value(right_grp10[1], right_grp10[2])}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(right_grp[0], right_grp[1], feature_cols, right_grp[2])
print(f'	2:f{fidx}<={thres:.8f}], gain={gain}')
print(f'		5:leaf={leaf_value(left_grp10[1], left_grp10[2])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols, right_grp10[2])
print(f'		6:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			13:leaf={leaf_value(left_grp101[1], left_grp101[2])}')
print(f'			14:leaf={leaf_value(right_grp101[1], right_grp101[2])}')

0:[f22<=102.65000000], gain=106.49711247810995
	1:[f24<=0.17825000], gain=4.100084403469467
		3:[f12<=4.11850000], gain=2.700040361807069
			7:leaf=0.36628723424908344
			8:leaf=-0.0686659774651944
		4:leaf=-0.14008902900334624
	2:f7<=0.04886500], gain=13.117061434887773
		5:leaf=0.06373397473645441
		6:[f21<=20.64500000], gain=5.665207541900578
			13:leaf=-0.009061999925708131
			14:leaf=-0.4306924605015128
